# 1. RNN의 한계
## 1.1 Attention의 등장
- Attention의 등장 배경: Seq2Seq의 병목현상
    - 고정된 크기의 하나의 컨택스트 벡터에 모든 정보를 압축하면서 정보 손실 문제가 발생함
    - 게다가 컨텍스트 벡터는 인코더의 마지막 hidden state이므로 마지막 단어의 정보는 선명한 반면 앞쪽 단어 정보는 여러 단계를 거치며 희미해짐

- **Attention** : 디코더가 단어를 생성할 때마다 **입력 문장 전체를 다시 보고**, 관련 있는 부분에 집중하여 **출력마다 새로운 Context Vector를 생성함.**
    - 출력 단어를 생성할 때 입력 문장 속 단어들 중 어떤 단어에 더 집중할지의 정도를 결정함

- **Attention은 무엇을 하는가?**
    - 예시 문장: 나는 사과를 먹었다. 그것은 맛있었다.
    - 여기서 '그것'이 무엇을 가리키는지 이해하려면 '사과'와의 관계를 살펴봐야함.
    - Attention은 문장 속 단어들이 서로 얼마나 관련 있는지를 계산하고, 어떤 단어에 집중할지 결정하는 과정이다.


- Attention은 디코더와 인코더의 모든 위치를 직접 비교하기 때문에, 각 단어와의 관련도 점수를 **동시에 계산할 수 있음 -> 병렬화 가능성**
    - but, 여전히 LSTM으로 한 칸 씩 순서대로 처리되기 때문에 모델 전체를 병렬화 할 순 없음<br> LSTM을 걷어내고 어텐션만 남긴 모델이 **Transformer!**

- **Attention의 작동 과정**
    1. 입력을 벡터 형태로 변환 - 컴퓨터가 읽을 수 있도록 숫자로 이루어진 벡터로 변환
    2. 각 단어의 중요도 계산
    - 관련도 점수: 현재 작업과 각 단어 사이의 관계가 얼마나 관련이 높고 유사한지 점수를 매김
    - Softmax : 관련도 점수를 0~1 사이의 확률 값으로 변환. 즉, 모든 단어의 중요도 합이 1인 비율로 바꿔주며 이 Softmax를 거친 비율을 Attention Weight(어텐션 가중치)라고 함.
    - 예시) '사과'가 관련도 점수 3.8점을 받았고, softmax이후 어텐션 가중치가 80% -> 이번 출력을 만들 때 '사과'의 정보를 80% 비중으로 반영한다는 뜻

- Attention 매커니즘의 장점
    1. 시간 경과에 따른 유연성
    - RNN의 경우, 멀리 떨어진 단어일 수록 정보가 흐려지는 **장기 의존성 문제**가 있음
    - 어텐션은 모든 단어를 직접 비교하므로 거리 상관없이 관계 파악이 가능함.
    2. 공간에 대한 유연성
    - CNN의 경우 고정된 크기의 필터로 **주변 정보만 봄**
    - 어텐션은 모든 위치를 직접 연결하여 **한 번에 전역적인 관계를 학습**할 수 있음
    3. 병렬화
    - RNN은 앞 단계가 끝나야 다음 단계를 계산하는 순차 구조라 학습이 느림
    - 어텐션은 관련도 계산이 각 단어 쌍마다 서로 독립적이므로 동시에 계산이 가능함

## 1.2 Attention is all you need
- Transformer 구조를 처음 발표한 논문 (Self-Attention 방식) -> 제목 그대로 “Attention만 있으면 된다”는 주장을 담은 논문

- 논문 주요 내용
    1. CNN과 RNN을 제거하고 **self-attention** 으로 입력과 출력의 관계를 계산하는 **Transformer 모델**을 제안
    2. 순서가 있는 데이터를 처리하기 위해서 **Recurrence(순환)와 Convolution(합성곱)은 필수가 아니다** <br>->순차 구조를 없앤 덕분에 병렬화를 통한 빠른 학습 속도가 가능해짐

- 파급효과 : LLM 탄생, NLP 평정, 이미지 및 음성 등 타 분야로 확장

- **Transformer란?** : RNN처럼 단어를 순서대로 처리하지 않고 **Attention만으로 문장 전체의 의미와 구조를 파악하여 한 번에 처리**하는 인코더-디코더 모델

- 기존 모델과 Transformer의 차이: 인코더와 디코더 내부를 무엇으로 채워 구조를 만들었는가

    | 구분 | Seq2Seq + Attention | Transformer |
    |---|---|---|
    | 인코더, 디코더의 내부 | LSTM (순차 처리) | Self-Attention |
    | 어텐션의 역할 | RNN을 돕는 보조 역할 | 모델 그 자체 |
    | 어텐션 사용 위치 | 디코더 → 인코더 1곳 | 인코더 내부, 디코더 내부, 인코더-디코더 사이 3곳 |
    | 단어 순서 정보 | RNN이 순서대로 읽으며 반영 | Positional Encoding으로 따로 추가 |
    | 병렬 처리 | 불가 | 가능 |

# 2. Self-Attention
## 2.1 Query, Key, Value
- LSTM과 달리 Transformer는 모든 단어를 직접 비교하는데 이때 쓰이는 재료가 **Query, Key, Value**임

- Query: 현재 처리 중인 단어가 다른 단어들에게 “지금 나한테 필요한 정보가 뭐지?”라고 묻는 벡터
- Key: 각 단어가 “나는 이런 정보를 가지고 있어” 라고 보이는 벡터. 즉, 키는 검색 대상이 되는 모든 정보의 ‘이름표’
- Value: 각 단어가 실제로 전달하는 정보 벡터. 관련도 비율만큼 모든 Value를 섞어 최종 결과를 만듦

- 유튜브 예시
    - Q (Query, 쿼리): 내가 검색창에 친 검색어 → "지금 나는 "학생" 정보가 필요해"
    - K (Key, 키): 각 영상의 제목·태그 → ‘학교 생활’, ‘교육학 개론’, ‘요리 레시피’, ‘주식 투자’ 제목표 → ‘학생’과 비교하여, 60%, 35%, 3%, 2%
    - V (Value, 밸류): 영상의 실제 내용. 학교 생활 책 내용 60% + 교육학개론 35% + 요리 레시피 3% + 주식투자 2% → 최종 답변


## 2.2 기존 Cross Attention vs Self Attention 
- 핵심: 기존 Seq2Seq에서 attention은 디코더가 인코더를 참고하는 Cross Attention이라면, <br>Transformer는 **문장이 자기 스스로를 참고하는 Self Attention**

- **Cross Attention**: Query와 Key-Value쌍이 **서로 다른 시퀀스에서 출력**<br>-> 디코더가 인코더에게 “지금 어디를 봐야 하지?”라고 묻는 참고 구조

- 핵심 과정 예시
    - 다른 시퀀스 (인코더 쪽): 입력 문장 "나는 / 사과를 / 먹었다"
    - 하나의 시퀀스 (디코더 쪽): 지금 만들고 있는 문장 “I / ate / …”
    1. 디코더 쪽에서 Query 만듦 -> “다음 단어를 위해 어떤 영어 단어를 봐야 할까?”
    2. 인코더 쪽에서 K,V 만듦 -> 인코더가 원문의 각 단어 (I,ate,an,apple)에 대해 만들어 둔 벡터를 Key와 Value로 사용
    3. 유사도 계산 -> Query와 각 Key의 관련도를 계산하고 Softmax로 확률의 값 비율로 변환
    4. Weighted Sum -> 비율만큼 Value를 섞어 Context Vector 생성 → 디코더 자기 정보 + Context Vector로 다음 단어 출력

- **Self Attention** : Query와 Key-Value 쌍이 **같은 시퀀스에서 출력**<br>문장이 스스로에게 “우리끼리 누가 누구와 관련이 있지?”라고 묻는 구조

- 핵심 과정 예시
    - 예시 : “가을 은행 냄새 지독해”에서 ‘은행’의 의미를 파악해보자
    1. 벡터 임베딩 - 각 단어를 컴퓨터가 읽을 수 있도록 벡터로 변환하고 같은 벡터에서 Q, K, V를 만듦
    2. 내적 및 관계도 계산 - ‘은행’의 Query와 모든 단어의 Key를 비교해 관계 점수 계산
    3. 중요도 배분 (Softmax & 어텐션 가중치) - 가을 30%, 은행 20%, 냄새 45%, 지독해 5%로 비율 변환 및 특정 단어를 이해에 대한 각각의 %로 중요도 판단
    4. 새로운 벡터 생성 - 비율만큼 Value를 섞어 "가을에 냄새 나는 열매"라는 의미를 담은 새 벡터가 생성

⇒ 이 과정을 모든 단어가 동시에 거쳐 각자 문맥이 반영된 새 벡터를 얻음 → **병렬 처리 가능**


## 2.3 Scaled Dot-Product Attention
- Dot-Product Attention: Query와 Key 비교를 통해 관련도 계산하는 과정에서, '비교'는 Dot-Product Attention의 계산 방법으로 이루어짐

- Dot-Product Attention의 과정
    1. Attention Score 계산 - Query와 Key 벡터를 내적하여 관련도 점수 구함
    2. 계산된 값에 Softmax 적용 -> Attention weight 산출
        - 점수표의 각 가로줄을 합이 1인 비율로 전환. “이 단어는 다른 단어들을 몇 %씩 참고하는가”를 나타냄
    3. 최종 결과값 계산 - 새 단어 벡터 = Σ (그 단어가 각 단어에 준 어텐션 비율 × 그 단어의 Value)<br>예시: 새 "은행" = Σ ("은행"이 각 단어에 준 비율 × 각 단어의 Value)
    

- 문제: Attention Score의 극단화
    - 벡터 차원이 커질수록 내적(Dot Product) 값이 지나치게 커질 수 있음 <br>-> 이러한 극단적인 값이 Softmax 함수에 입력되면 특정 단어에 확률이 과도하게 몰리는 현상이 발생 <br>-> 이로 인해 기울기가 0에 가까워져 학습이 원활하게 진행되지 않는 기울기 소실 문제 발생 가능

- **Scaled Dot-Product Attention** : 기존의 Dot-Product Attention 문제를 해결한 방법 -> 내적과 Softmax 사이에 **스케일링 단계 하나를 추가**

- 과정
    - Query와 모든 Key를 내적 → Attention Score
    - Attention Score를  $$\sqrt{d_k}$$로 나누는 스케일링 과정 진행 
        - d_k = Key(그리고 Query) 벡터의 길이(차원 수)
    - Softmax 적용 → Attention Weight 산출
    - Attention Weight x Value → 최종 결과

- 루트 d_k 로 나누면 분산이 다시 1 근처로 돌아와, 차원의 크기와 상관없이 점수가 안정적인 범위에 머물 수 있게됨.

# 3. Multi-Head Attention
- Head란? : **문장을 서로 다른 관점**으로 분석할 수 있는데, **각각 다른 관점으로 Attention을 계산하는 단위가 Head!**
    - 예시: "나는 사과를 먹었다. 그것은 맛있었다"
    - Head1 : 지시 관계에 집중
    - Head2 : 행동 관계에 집중
    - Head3 : 의미 관계에 집중

- 헤드마다 가중치 행렬의 숫자가 다르기 때문에, 같은 문장을 봐도 집중하는 단어가 달라짐

- Single Head: 헤드가 1개 
    - 가중치 행렬 세트 하나로, 512차원 벡터를 그대로 한번 Attention함
    - **한계**: 한 단어를 이해하려면 여러 종류의 관계를 동시에 고려해야 하는데, 헤드가 하나 뿐이면 하나의 관점으로만 Attention을 계산함
    - 예시 : “Tom gave Mary a book yesterday”

    | **단어** | **Tom** | **Mary** | **a** | **book** | **yesterday** |
    |---|---:|---:|---:|---:|---:|
    | **“gave”의 가중치** | 0.25 | 0.25 | 0.02 | 0.25 | 0.23 |
    - 네 단어가 다 중요하니 비슷하게 나눠 줄 수 밖에 없음 -> 준 사람이 Tom인지 Mary인지 흐려짐

- **Multi-Head Attention**: Attention을 작게 쪼개서 여러 번 동시에 하는 방식

    | **헤드** | **주로 보는 단어** | **가중치** | **담당하는 질문** |
    |---|---|---:|---|
    | 헤드 1 | Tom | 0.90 | Who? |
    | 헤드 2 | Mary | 0.90 | To whom? |
    | 헤드 3 | book | 0.90 | What? |
    | 헤드 4 | yesterday | 0.90 | When? |
- 헤드마다 질문 하나에 집중하고, 결과를 나란히 이어 붙이기 때문에 네 정보가 섞이지 않고 또렷하게 남음

## 3.1 Multi-Head Attention 작동 
- 예시: "I love you" 에서 "love" 한 단어가 Multi Head Attention을 통과하는 과정을 알아보기
- 각 단어의 차원: 512, 헤드 수:8개, 각 헤드의 차원: 64차원 (512/8)

1. 분할- 8개의 관점 만들기
    | **헤드** | **Query** | **Key** | **Value** |
    |---|---|---|---|
    | 헤드 1 | love × `W_1^Q` → 64차원 | love × `W_1^K` → 64차원 | love × `W_1^V` → 64차원 |
    | 헤드 2 | love × `W_2^Q` → 64차원 | love × `W_2^K` → 64차원 | love × `W_2^V` → 64차원 |
    | … | … | … | … |
    | 헤드 8 | love × `W_8^Q` → 64차원 | love × `W_8^K` → 64차원 | love × `W_8^V` → 64차원 |
2. 병렬 어텐션 계산- 각 헤드별 각자 분석
    - 8개 헤드가 서로 간섭하지 않고 동시에 Scaled Dot-Product Attention을 계산
3. 이어 붙이기
    - 8개의 64차원 결과를 순서대로 한 줄로 이어 붙여 512차원(8×64) 벡터를 하나 만듦
4. 섞기
    - 이어 붙인 벡터는 아직 각 헤드의 분석 결과를 단순히 나열한 것이기 때문에 새로운 가중치 행렬 W0을 곱해서 **8개 관점을 하나의 의미로 융합**
        - 작동 예시: W^O는 학습을 통해 값이 결정되는 새로운 가중치 행렬.
        - 이 행렬을 곱하면 일어나는 일: 
        - Head 1의 정보와 Head 2의 정보를 조합  
        - Head 2의 정보와 Head 3의 정보를 조합
        - 여러 헤드의 정보를 다양한 비율로 조합

⇒ 모든 단계가 끝나면 ‘love’라는 단어에 대한 **Multi-Head Attention 레이어의 최종 출력 벡터**를 얻게 됨.

# 4. Transformer의 전체 아키텍처
## 4.1 Transformer의 전처리 단계
- 한 눈에 보기: 텍스트 입력(Inputs) → 텍스트 토큰화 (Tokenization) → 토큰 임베딩 (Embedding) → Positional Encoding 순으로 동작

**각 단계 세부 설명**
1. 토큰화 - 문장을 모델이 다루는 기본단위로 조각내는 것
    - 보통 단어 하나 = 토큰 하나: Give, me, an ,apple
    - 문장 부호도 토큰임
    - 길거나 드문 단어는 여러 조각: didn't → did + n't

2. 임베딩 - 임베딩 과정은 각 토큰을 숫자(벡터)로 바꾸는 것
    - 임베딩의 원리: **단어에 좌표를 부여하여 공간적 의미로 표현하는 것**
    - 핵심원리: **비슷한 단어는 가까운 좌표에 놓인다**
    - 임베딩 값은 처음엔 무작위이고, 학습하면서 점점 의미에 맞는 자리를 찾아감.
    - 이처럼 단어마다 기본적인 의미가 담긴 덕분에, 이후 Attention이 단어 사이의 관계를 계산할 수 있음

3. Positional Encoding - 단어들의 순서 정보를 임베딩 벡터에 추가하는 단계
    - Transformer는 모든 단어를 한꺼번에 보기 때문에, **단어 순서를 모름**
    - 각 위치마다 정해진 512차원 '위치 벡터'를 만들어 임베딩에 더함.
    - 예시: "나는 너를 사랑해"
        - 나는 + 위치1(100)
        - 너를 + 위치2(200)
        - 사랑해 + 위치3(300)

## 4.2 Transformer의 Encoder와 Decoder
- **Encoder 층** : Multi-Head Self-Attention과 Feed-Forward Network로 이루어짐
    1. Multi-Head Self-Attention: 원문 단어끼리 서로를 보며 문맥 파악

    2. Feed-Forward Network: 단어 하나하나를 따로 더 깊이 가공

- Feed-Forward Network란?: 
    - 512차원을 2048차원으로 넓혔다가 다시 512차원으로 좁힘. 그 결과 차원을 확장하여 다양한 특징을 표현할 수 있음
    - 각 단어에 FFN을 개별적으로 적용하지만, 세 단어 모두 동일한 가중치를 사용 ->이 단계에서는 단어끼리 정보를 주고 받지 않음

- Add & Norm: Multi Head Attention과 FFN에 모두 붙는 안전장치
    - Add = Residual Connection(잔차연결)
        - 부품을 통과한 결과에 원래 입력을 그대로 더해줌

    - Norm = Layer Normalization (층 정규화)
        - 한 단어 벡터안의 512개 숫자를 평균 0, 분산 1 근처로 맞춰줌.

- **Decoder 층** : Masked Multi-Head Self-Attention, Encoder-Decoder Attention (Cross-Attention), Feed-Forward Network로 이루어짐
    1. Masked Multi-Head Self-Attention: 지금까지 쓴 번역 단어끼리 관계 파악 (미래 단어는 가림)
    
    2. Encoder-Decoder Attention (Cross-Attention): 원문 분석 자료(Encoder 출력)에서 필요한 부분 찾기
    
    3. Feed-Forward Network:Encoder와 동일

    **번역문을 한 단어씩 쓰는 작성팀이라고 생각하기**

- Masked Multi-Head Attention: Decoder의 Self-Attention에서 **미래 시점의 단어 정보를 가려(Softmax 후 0) 자기보다 앞의 단어만 보게 하는 메커니즘**

- Encoder-Decoder Attention(Cross-Attention): 출력 문장 (Decoder)과 입력 문장 (Encoder)을 연결하는 과정
    - Q,K,V의 출처
        - Q = 직전 Decoder 서브레이어의 출력
        - K·V = Encoder 마지막 층의 출력 (6개 디코더 레이어가 모두 같은 Encoder 출력을 씀)
    - 즉, Encoder가 입력 문장을 한 번 분석해서 최종 결과를 만들면, Decoder의 6개 레이어가 그 결과를 공통 자료(K, V)로 활용하여 출력 문장을 생성함.